In [ ]:
# 查询ifcSpatialZone以及extended parameters
import ifcopenshell
import ifcopenshell.util.element

# 1. 载入模型 (请替换为你的实际路径)
model_path = "C:/Users/lanxuan/model package/IFC-Tunnel-Legal-Extension_TEST.ifc" 
model = ifcopenshell.open(model_path)
spatial_zones = model.by_type("IfcSpatialZone")

# 2. 🎯 定义你的“狙击目标列表”
# 将你在 Notepad++ 里发现的核心自定义属性名填在这里
target_properties = [
    "EasementOwner",
    "EasementState",
    "EasementType",
    "ParcelID",
    "ParcelType"
]

# 为了防止大小写拼写导致漏查，我们把目标全转成小写作为比对基准
target_props_lower = [p.lower() for p in target_properties]

print("🔍 正在启动精准雷达，扫描包含指定地籍属性的空间...\n")
print("=" * 60)

valid_zone_count = 0 # 计数器

# 3. 开始遍历过滤
for zone in spatial_zones:
    psets = ifcopenshell.util.element.get_psets(zone)
    found_custom_props = {} # 临时口袋，专门装找到的目标属性
    
    for pset_name, properties in psets.items():
        if isinstance(properties, dict):
            for prop_name, prop_value in properties.items():
                
                # 🚨 终极拦截器：直接比对属性名！
                # 只要这个属性的名字在我们的“狙击列表”里，不管它是什么数据类型，全部抓取！
                if prop_name.lower() in target_props_lower:
                    found_custom_props[prop_name] = prop_value
    
    # 4. 判定：如果口袋里装到了目标属性，才输出这个空间！
    if found_custom_props:
        valid_zone_count += 1
        zone_name = zone.Name if zone.Name else "未命名空间"
        print(f"🟦 目标空间: {zone_name} (GlobalId: {zone.GlobalId})")
        
        # 提取所在楼层
        container = ifcopenshell.util.element.get_container(zone)
        if container:
            print(f"  📍 所在层级: {container.Name}")
        else:
            print("  📍 所在层级: 未明确")

        # 仅仅输出你关心的那几个自定义属性
        print("  📝 提取到的核心地籍属性:")
        for p_name, p_val in found_custom_props.items():
            print(f"    - {p_name} : {p_val}")
            
        print("-" * 60)

print(f"\n🎉 筛选完毕！共有 {valid_zone_count} 个空间包含了你指定的地籍属性。")

In [3]:
# Model line - ifcAnnotation to ifcAlignment 并保留
import ifcopenshell
import ifcopenshell.api

# 1. 加载包含多个 IfcAnnotation (Model Line) 的 IFC 文件
model_path = "C:/Users/lanxuan/WGTP/base model/Underground_drawing_0606_integrated.ifc" # 请确保路径正确
model = ifcopenshell.open(model_path)

print("🔍 正在寻找特定的 Model Line (ID: 418840)...")
annotations = model.by_type("IfcAnnotation")
target_line = None

# 2. 核心过滤逻辑：使用 getattr 探测器
for anno in annotations:
    anno_name = getattr(anno, "Name", "")
    anno_desc = getattr(anno, "Description", "")
    
    if "418840" in str(anno_name) or "418840" in str(anno_desc):
        target_line = anno
        break 

if not target_line:
    print("❌ 未找到 ID 为 418840 的 IfcAnnotation，请检查文件。")
else:
    print(f"✅ 成功锁定目标路线，IFC GlobalId: {target_line.GlobalId}")

    # 3. 创建一个合法的 IfcAlignment 实体
    print("✨ 正在生成 IfcAlignment 基础设施路线...")
    alignment = ifcopenshell.api.run("root.create_entity", model, ifc_class="IfcAlignment", name="SpatialAlignment")

    # 4. 几何体“复制” (共享原线条的空间位置和 3D 形状)
    alignment.ObjectPlacement = target_line.ObjectPlacement
    alignment.Representation = target_line.Representation

    # 5. 将新的 Alignment 挂载到场地 (IfcSite) 上
    site = model.by_type("IfcSite")[0]
    ifcopenshell.api.run("aggregate.assign_object", model, relating_object=site, products=[alignment])

    # 注意：model.remove(target_line) 已被移除。IfcAnnotation 将被安全保留。
    print("✨ IfcAlignment 生成完毕，同时保留了原始的 IfcAnnotation 以供可视化查看。")

    # 6. 保存终极模型
    output_path = "C:/Users/lanxuan/WGTP/testifcmodel/WGTP_Alignment_Dual.ifc"
    model.write(output_path)
    print(f"🎉 转换成功！新模型已保存至: {output_path}")

🔍 正在寻找特定的 Model Line (ID: 418840)...
✅ 成功锁定目标路线，IFC GlobalId: 0kKbgSRZLBLfhfiOt0D99f
✨ 正在生成 IfcAlignment 基础设施路线...
✨ IfcAlignment 生成完毕，同时保留了原始的 IfcAnnotation 以供可视化查看。
🎉 转换成功！新模型已保存至: C:/Users/lanxuan/WGTP/testifcmodel/WGTP_Alignment_Dual.ifc


In [4]:
# legal and physical integration
import ifcopenshell
import uuid

def new_guid():
    return ifcopenshell.guid.compress(uuid.uuid4().hex)

# 1. 打开你准备好的 IFC 文件
ifc_file = ifcopenshell.open(r"C:\Users\lanxuan\WGTP\testifcmodel\WGTP_Alignment_Dual.ifc")

# 2. 抓取三类核心实体
spatial_zones = ifc_file.by_type("IfcSpatialZone")
civil_elements = ifc_file.by_type("IfcCivilElement")
alignments = ifc_file.by_type("IfcAlignment") # 新增：抓取路线骨架

print(f"🔍 Found {len(spatial_zones)} IfcSpatialZone instances")
print(f"🔍 Found {len(civil_elements)} Physical Elements instances")
print(f"🔍 Found {len(alignments)} IfcAlignment instances")

# 3. 将物理管片和空间路线合并为一个“待关联列表” (核心修改点)
elements_to_link = civil_elements + alignments

owner_history = ifc_file.by_type("IfcOwnerHistory")[0]

# 4. 循环为每个法律空间建立关联
for zone in spatial_zones:
    rel = ifc_file.createIfcRelReferencedInSpatialStructure(
        GlobalId=new_guid(),
        OwnerHistory=owner_history,
        Name=f"LegalPhysicalLink_{zone.Name}",
        # 描述中明确注明包含了 Alignment
        Description="Legal-physical linkage between legal space, physical elements, and tunnel alignment",
        RelatedElements=elements_to_link, # 将合并后的列表一次性注入
        RelatingStructure=zone
    )
    print(f"✅ Created comprehensive relationship for zone: {zone.Name}")

# 5. 保存终极模型
output_path = r"C:/Users/lanxuan/WGTP/testifcmodel/Integration_fully_linked.ifc"
ifc_file.write(output_path)
print(f"🎉 Done. Saved to: {output_path}")

🔍 Found 6 IfcSpatialZone instances
🔍 Found 798 Physical Elements instances
🔍 Found 1 IfcAlignment instances
✅ Created comprehensive relationship for zone: Space 48:599978
✅ Created comprehensive relationship for zone: Space 58:601619
✅ Created comprehensive relationship for zone: Space 62:601988
✅ Created comprehensive relationship for zone: Space 63:602300
✅ Created comprehensive relationship for zone: Space 64:604795
✅ Created comprehensive relationship for zone: Space 65:604802
🎉 Done. Saved to: C:/Users/lanxuan/WGTP/testifcmodel/Integration_fully_linked.ifc
